In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# Install the core framework, AI connectors, and reporting tools.
!pip install -qU langgraph langchain_community tavily-python
!pip install -qU langchain-tavily
!pip install -qU langchain-groq
!pip install -qU fpdf2 markdown python-dotenv

In [ ]:
"""Multi-Agent Research System — LangGraph prototype.

Demonstrates a four-stage pipeline:
    1. SearchAgent    — expand topic into queries, fetch live web data
    2. SynthesisAgent — group evidence into themes, detect contradictions
    3. ReportAgent    — generate a structured Markdown report
    4. Evaluator      — score report quality on a rubric
"""

from __future__ import annotations

import json
import logging
import os
import re
from typing import Any

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from tavily import TavilyClient

# ── Configuration ─────────────────────────────────────────────────────────────
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger(__name__)

# ── LLM & Search Clients ──────────────────────────────────────────────────────
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

search_tool = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

logger.info("Setup complete: LLM and search tool are connected.")

In [ ]:
# ── Shared State ──────────────────────────────────────────────────────────────
from pydantic import BaseModel, Field


class ResearchState(BaseModel):
    """Shared state passed between all agents in the pipeline."""

    task: str
    queries: list[str] = Field(default_factory=list)
    sources: list[dict[str, Any]] = Field(default_factory=list)
    evidence_clusters: dict[str, Any] = Field(default_factory=dict)
    contradictions: list[str] = Field(default_factory=list)
    report: str = ""
    log: list[str] = Field(default_factory=list)

In [ ]:
# ── Agent 1: Search ───────────────────────────────────────────────────────────

def search_agent(state: ResearchState) -> dict[str, Any]:
    """Expand the research task into queries and fetch live web sources.

    Stages:
        1. Ask the LLM to generate 3 distinct search queries for the task.
        2. Run each query through Tavily and collect all results.

    Returns a partial state update with queries, sources, and a log entry.
    """
    logger.info("Searching for sources …")

    query_prompt = (
        f"You are a Lead Researcher. Given the topic: {state.task}, "
        "generate 3 distinct search queries. Output ONLY a JSON list of strings."
    )
    response = llm.invoke(query_prompt)

    try:
        json_match = re.search(r"\[.*\]", response.content, re.DOTALL)
        queries: list[str] = json.loads(json_match.group(0)) if json_match else []
    except Exception as exc:
        logger.warning("Failed to parse query JSON: %s", exc)
        queries = [state.task]

    all_results: list[dict[str, Any]] = []
    for q in queries:
        results = search_tool.search(query=q)["results"]
        all_results.extend(results)

    logger.info("Found %d sources.", len(all_results))
    return {
        "queries": queries,
        "sources": all_results,
        "log": state.log + [f"Found {len(all_results)} sources."],
    }

In [ ]:
# ── Agent 2: Synthesis ────────────────────────────────────────────────────────

def synthesis_agent(state: ResearchState) -> dict[str, Any]:
    """Group search results into thematic clusters and flag contradictions.

    Stages:
        1. Build a context string from all fetched sources.
        2. Ask the LLM to return themes and contradictions as JSON.

    Returns a partial state update with evidence clusters, contradictions,
    and a log entry.
    """
    logger.info("Synthesizing evidence …")

    context = "\n".join(
        f"Source: {s.get('url')}\nContent: {s.get('content')}"
        for s in state.sources
    )
    synthesis_prompt = (
        f"Analyze these regarding '{state.task}':\n"
        "1. Group into themes. 2. Identify DISAGREEMENTS.\n"
        "Output ONLY valid JSON with keys: 'themes' (dict) and 'contradictions' (list).\n"
        f"Context: {context}"
    )
    response = llm.invoke(synthesis_prompt)

    try:
        json_match = re.search(r"\{.*\}", response.content, re.DOTALL)
        analysis: dict[str, Any] = (
            json.loads(json_match.group(0))
            if json_match
            else {"themes": {}, "contradictions": []}
        )
    except Exception as exc:
        logger.warning("Failed to parse synthesis JSON: %s", exc)
        analysis = {
            "themes": {"Summary": "Error parsing JSON — raw text summary unavailable."},
            "contradictions": [],
        }

    return {
        "evidence_clusters": analysis.get("themes", {}),
        "contradictions": analysis.get("contradictions", []),
        "log": state.log + ["Synthesized evidence into clusters."],
    }

In [ ]:
# ── Agent 3: Report ───────────────────────────────────────────────────────────

def report_agent(state: ResearchState) -> dict[str, Any]:
    """Transform synthesised evidence into a structured Markdown report.

    The report follows this structure:
        1. Title  2. Introduction  3. Thematic Findings (with citations)
        4. Discussion of Agreements/Disagreements  5. Limitations
        6. Conclusion  7. References

    Returns a partial state update with the generated report and a log entry.
    """
    logger.info("Generating final research report …")

    themes_data = json.dumps(state.evidence_clusters, indent=2)
    contradictions_data = "\n".join(state.contradictions)
    source_list = "\n".join(f"- {s.get('url')}" for s in state.sources)

    report_prompt = (
        "You are a Professional Research Writer. "
        f"Create a comprehensive report for: {state.task}\n\n"
        f"THEMES: {themes_data}\n"
        f"CONTRADICTIONS: {contradictions_data}\n"
        f"SOURCES: {source_list}\n\n"
        "STRICT STRUCTURE:\n"
        "1. Title  2. Introduction  3. Thematic Findings (with [Source URL] citations)\n"
        "4. Discussion of Agreements/Disagreements  5. Limitations  6. Conclusion  7. References\n\n"
        "Format the entire response in clean, professional Markdown."
    )
    response = llm.invoke(report_prompt)

    return {
        "report": response.content,
        "log": state.log + ["Generated full structured report with citations."],
    }

In [ ]:
# ── Pipeline Assembly ─────────────────────────────────────────────────────────
from langgraph.graph import END, StateGraph

workflow: StateGraph = StateGraph(ResearchState)

# Nodes — one per agent
workflow.add_node("searcher", search_agent)
workflow.add_node("synthesizer", synthesis_agent)
workflow.add_node("writer", report_agent)

# Edges — linear sequential flow
workflow.set_entry_point("searcher")
workflow.add_edge("searcher", "synthesizer")
workflow.add_edge("synthesizer", "writer")
workflow.add_edge("writer", END)

pipeline = workflow.compile()
logger.info("Pipeline compiled — multi-agent system is ready.")

In [ ]:
# ── Execution ─────────────────────────────────────────────────────────────────
from IPython.display import Markdown

research_task = "What are the main trade-offs between CNNs and Vision Transformers for medical imaging?"

final_state = pipeline.invoke({"task": research_task})

Markdown(final_state["report"])

In [ ]:
# ── Evaluator ─────────────────────────────────────────────────────────────────

def self_evaluation(report: str, task: str) -> str:
    """Score a research report on a four-criterion rubric using the LLM.

    Rubric:
        1. Coverage      (1x10) — addresses key parts of the question
        2. Faithfulness  (1x10) — claims supported by provided sources
        3. Hallucination (1x10, lower is better) — unsupported assertions
        4. Usefulness    (1x10) — decision-supportive quality

    Returns the raw evaluation text from the LLM.
    """
    eval_prompt = (
        f"Evaluate this research report on '{task}' using this rubric:\n"
        "1. Coverage (1-10): Did it address key parts of the question?\n"
        "2. Faithfulness (1-10): Are claims supported by the provided sources?\n"
        "3. Hallucination Rate (1-10, lower is better): Are there unsupported claims?\n"
        "4. Usefulness (1-10): Is it decision-supportive?\n\n"
        f"REPORT: {report}"
    )
    return llm.invoke(eval_prompt).content


evaluation = self_evaluation(final_state["report"], research_task)
print(evaluation)

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
from markdown import markdown


_HTML_STYLE = """
<style>
  body        { font-family: sans-serif; line-height: 1.6; max-width: 800px; margin: auto; padding: 40px; color: #333; }
  h1          { color: #2c3e50; border-bottom: 2px solid #eee; }
  h2          { color: #34495e; margin-top: 30px; }
  pre         { background: #f4f4f4; padding: 10px; border-radius: 5px; overflow-x: auto; }
  blockquote  { border-left: 5px solid #eee; padding-left: 20px; font-style: italic; }
</style>
"""


def export_to_html(report_markdown: str, output_path: str = "Research_Report.html") -> None:
    """Convert a Markdown report to a styled HTML file and write it to disk.

    Args:
        report_markdown: The Markdown content to convert.
        output_path: Destination file path for the HTML output.
    """
    html_content = _HTML_STYLE + markdown(report_markdown)
    with open(output_path, "w", encoding="utf-8") as fh:
        fh.write(html_content)
    logger.info("HTML report written to %s", output_path)


export_to_html(final_state["report"], "Medical_Imaging_Research.html")